In [8]:
import pandas as pd
from google.oauth2 import service_account
from googleapiclient.discovery import build
import os
import json

from dotenv import load_dotenv


In [15]:
# Load ENV vars
load_dotenv(dotenv_path='../.env')
SERVICE_ACCOUNT = os.getenv('SERVICE_ACCOUNT')
SPREADSHEET_ID = os.getenv('SPREADSHEET_ID')

In [17]:
# Parse the credentials from the environment variable
credentials_dict = json.loads(SERVICE_ACCOUNT)
credentials = service_account.Credentials.from_service_account_info(
    credentials_dict, 
    scopes=['https://www.googleapis.com/auth/spreadsheets']
)


In [18]:
# Initialize the Google Sheets API
sheets_client = build('sheets', 'v4', credentials=credentials)

In [19]:
def fetch_sheet(sheet_name, col_start="B", col_end="B", row_start=2, row_end=200):
    
    # Create the range
    range_ = f"{sheet_name}!{col_start}{row_start}:{col_end}{row_end}"

    # Fetch the data from the Google Sheets API
    response = sheets_client.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_
    ).execute()

    # Get the values from the response
    values = response.get('values', [])

    # Convert the fetched data into a pandas DataFrame
    df = pd.DataFrame(values[1:], columns=values[0])  # values[0] is the header row

    return df

In [24]:
df_mt = fetch_sheet('Macro-Territories', col_end='C')
df_tr = fetch_sheet('Territories', col_end='I')
df_ti = fetch_sheet('Tiers', col_end='C')
df_br = fetch_sheet('Brands', col_end='K')
df_pr = fetch_sheet('Products', col_end='K')

In [26]:
df_mt.columns

Index(['Macro-Territory Name', 'Description'], dtype='object')

In [49]:
def split_series(series, sep=' - '):
    return series.str.split(sep, expand=True).apply(lambda series: series.str.strip(), axis=0)

In [77]:
def split_string(str, sep=', '):
    return str.split(', ') if str else []

In [90]:
# Macro

columns = {
    'Macro-Territory Name': 'name',
    'Description': 'desc'
}

cl_mt = df_mt.rename(columns=columns)

split_name = split_series(cl_mt.name)

cl_mt['id'] = split_name[0]
cl_mt['name'] = split_name[1]

to_json = lambda d: {
   'id': d['id'],
    'data': { 
        'name': d['name'],
        'desc': d['desc']
    }
}

mt = [ to_json(d) for d in cl_mt.to_dict(orient='records')]

mt[0]

{'id': '1',
 'data': {'name': 'Superior Performance',
  'desc': 'Today, the majority of claims presented in the market are about superior performance, regardless of whether or not the brands are in this macro-territory. Therefore, it needs to account for a series of associations. The main association with superior performance is trust, in order to ensure the delivery of an expected result. This trust in superior performance can be approached in different ways by brands and the most relevant ones gave rise to brand activators and their respective territories.'}}

In [89]:
# Territory

columns = {
    'Territory Name': 'name',
    'Macro-Territory': 'macroId',
    'Description': 'desc',
    'What the territory activates': 'activates',
    'Types of Products': 'productTypes',
    'Trends': 'trends',
    'Product Claims': 'productClaims',
    'Images': 'img',
}

cl_tr = df_tr.rename(columns=columns)

split_name = split_series(cl_tr.name)
split_macro = split_series(cl_tr.macroId)

cl_tr['id'] = split_name[0]
cl_tr['name'] = split_name[1]
cl_tr['macroId'] = split_macro[0]


to_json = lambda d: {
    'id': d['id'],
    'macroId': d['macroId'],
    'data': {
        'desc': d['desc'],
        'activates': d['activates'],
        'productTypes': split_string(d['productTypes']),
        'trends': split_string(d['trends']),
        'productClaims': split_string(d['productClaims']),
        'pathImg': '/img/territory/' + d['img'] if d['img'] else None,
    }
}

tr = [ to_json(d) for d in cl_tr.to_dict(orient='records')]

tr[0]

{'id': '1',
 'macroId': '1',
 'data': {'desc': 'Conveys the message of superior performance and confidence through security/backing, which allows for acting/living with more freedom and boldness.',
  'activates': 'Activates freedom and the ability to go further. Activates the confidence that allows me to have the courage and boldness to live life to the fullest.',
  'productTypes': ['Good for my Machine',
   'Whiteness/Brightness',
   'Basic Cleaning',
   'Freshness',
   'Dirt and odor refresher',
   'Powerful stain remover',
   'Repellent and sun protection;',
   'Stain Removal',
   'Faster Stain Removal',
   'Stain Prevention',
   'Works with auto dose machine'],
  'trends': ['Home(care)tainment',
   'Fabric Care',
   'Unconventional materials',
   'Cleaning like a pro',
   'Holistic inclusivity'],
  'productClaims': [],
  'pathImg': '/img/territory/Freedom'}}

In [87]:
# Tier

columns = {
    'Macro-Territory': 'macroId',
    'Tier Name': 'id',
}

cl_ti = df_ti.rename(columns=columns)

cl_ti['id'] = split_series(cl_ti.id)[0]
cl_ti['macroId'] = split_series(cl_ti.macroId)[0]

to_json = lambda d: {
   'id': d['id'],
   'macroId': d['macroId']
}

ti = [ to_json(d) for d in cl_ti.to_dict(orient='records')]

ti[0]

{'id': '0', 'macroId': '1'}

In [ ]:
# Brands

columns = {
    'Territory Name': 'name',
    'Macro-Territory': 'macroId',
    'Description': 'desc',
    'What the territory activates': 'activates',
    'Types of Products': 'productTypes',
    'Trends': 'trends',
    'Product Claims': 'productClaims',
    'Images': 'img',
}
    